<a href="https://colab.research.google.com/github/guihahn/azuredev-38c0/blob/main/DIO_Traducao_Artigos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Modules

In [20]:
!pip install -q requests python-docx openai

## Translate Text

In [37]:
import requests
from docx import Document
import os

from google.colab import userdata

subscription_key = userdata.get('translatorSubscritpionKey')
# Correct endpoint for Azure Translator Text API
endpoint = 'https://api.cognitive.microsofttranslator.com'
region = 'eastus'
target_language = 'pt-br'

def translator_text(text, target_language=target_language):
    path= '/translate'
    constructed_url = endpoint + path
    headers = {
        'Ocp-Apim-Subscription-Key': subscription_key,
        'Ocp-Apim-Subscription-Region': region, # Required for Azure Cognitive Services
        'Content-type': 'application/json',
        'X-ClientTraceId': str(os.urandom(16))
    }
    body = [{
        'text': text
    }]

    params = {
        'api-version': '3.0',
        'from': 'en',
        'to': target_language
    }

    request = requests.post(constructed_url, params=params, headers=headers, json=body)
    response = request.json()
    # Extract and return the translated text
    if response and isinstance(response, list) and 'translations' in response[0] and response[0]['translations']:
        return response[0]['translations'][0]['text']
    else:
        return f"Translation failed: {response}"


In [30]:
translator_text("I know you're somewhere out there, somewhere far away")

'Eu sei que você está em algum lugar lá fora, em algum lugar bem longe'

### Translate Document

In [42]:
def translate_document(path):
    document = Document(path)
    full_text = []
    for paragraph in document.paragraphs:
        translated_text = translator_text(paragraph.text)
        full_text.append(translated_text)

    translated_doc = Document() # Create a new document to store translated content
    for line in full_text:
        translated_doc.add_paragraph(line)

    path_translated = path.replace('.docx', f'_{target_language}.docx')
    translated_doc.save(path_translated)

In [43]:
input_file = '/content/Talking_to_the_Moon.docx'
translate_document(input_file)